# G² Anahtarlık Çözümlemesi

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabricanatamanoder/KaraKitapIncelemesi/blob/main/g2_anahtarlik.ipynb)

Bu defter makalenin **Tablo 1** ve **Tablo 2**'sini ham veriden yeniden üretir.
Puan bantlarının her biri korpusun geri kalanıyla karşılaştırılır ve bantta
beklenenden çok geçen sözcükler listelenir.

Türkçe eklemeli bir dil olduğu için Türkçe korpusta sayım birimi sözlük
birimidir. Lemmalar Zemberek ile elde edilir. İngilizce korpusta sözcükler
metinde göründükleri hâlleriyle sayılır.

**Çalıştırma:** Çalışma zamanı → Tümünü çalıştır. Toplam süre bir dakika kadardır.

## 1. Zemberek kurulumu

In [ ]:
# Zemberek kurulumu. Bir kez çalışır, yaklaşık bir dakika sürer. ATLAMAYIN.
!pip install -q --no-deps zemberek-python
!pip install -q npy-append-array openpyxl

!pip download -q --no-binary :all: --no-deps "antlr4-python3-runtime==4.8" -d /tmp/a4
!cd /tmp/a4 && tar xzf antlr4-python3-runtime-4.8.tar.gz

import pathlib, shutil, site
kaynak = pathlib.Path('/tmp/a4/antlr4-python3-runtime-4.8/src/antlr4')

# Python 3.13'te typing.io kaldırıldı. antlr4 4.8 onu çağırdığı için iki dosya yamanır.
for f in kaynak.rglob('*.py'):
    s = f.read_text(encoding='utf-8')
    if 'typing.io' in s:
        f.write_text(s.replace('from typing.io import', 'from typing import'), encoding='utf-8')

hedef = site.getsitepackages()[0] + '/antlr4'
shutil.rmtree(hedef, ignore_errors=True)
shutil.copytree(kaynak, hedef)
print('kurulum tamam')

## 2. Veri

Veri dosyası depodan indirilir. Kendi verinizle çalışacaksanız `VERI_URL`
değerini değiştirin. Sütun adları README'deki tabloyla aynı olmalıdır.

In [ ]:
VERI_URL = 'https://raw.githubusercontent.com/sabricanatamanoder/KaraKitapIncelemesi/main/full_analysis_FINAL.xlsx'
DOSYA    = 'full_analysis_FINAL.xlsx'

import os, urllib.request
if not os.path.exists(DOSYA):
    urllib.request.urlretrieve(VERI_URL, DOSYA)
print('veri hazır:', DOSYA, os.path.getsize(DOSYA), 'bayt')

## 3. Ölçüt ve boru hattı

Bir sözcüğün listeye girebilmesi için G² değerinin 3,84'ü aşması gerekir
(p < 0,05). Ayrıca sözcük ilgili bantta en az üç kez ve en az üç farklı
yorumda geçmelidir. İkinci koşul, tek bir uzun yorumun listeyi doldurmasını
önler. İki eşiği de aşağıdan değiştirip etkisini görebilirsiniz.

In [ ]:
# -*- coding: utf-8 -*-
"""Tablo 1 ve Tablo 2 — G² ile ayırt edici sözcükler.
Türkçe tarafta Zemberek lemmatizasyonu, İngilizce tarafta yüzey biçimi.
"""
import collections, math, re, warnings, logging
warnings.filterwarnings('ignore'); logging.disable(logging.CRITICAL)

DOSYA         = 'full_analysis_FINAL.xlsx'
ASGARI_SIKLIK = 3    # sözcük ilgili bantta en az bu kadar geçmeli
ASGARI_YAYILIM= 3    # ve en az bu kadar FARKLI yorumda görünmeli
EN_UZUN       = 12   # her banttan alınacak sözcük sayısı

KINDLE = re.compile(r'your highlight|location \d|added on|kindle', re.I)
TOK_TR = re.compile(r"[a-zçğıöşüâîû]{3,}", re.I)
TOK_EN = re.compile(r"[a-z']{3,}", re.I)

DOMAIN_EN = {'book', 'books', 'read', 'reading', 'reader', 'readers'}

def durak_oku(ad):
    """Durak sözcük listesini depodan indirir."""
    import os, urllib.request
    if not os.path.exists(ad):
        urllib.request.urlretrieve(f'{DEPO_HAM}/{ad}', ad)
    return {s.strip() for s in open(ad, encoding='utf-8') if s.strip()}

DEPO_HAM = 'https://raw.githubusercontent.com/sabricanatamanoder/KaraKitapIncelemesi/main'

# Türkçe G² listesi kendi dosyasındadır. İngilizce G² listesi, bulut
# aşamasında kullanılan NLTK listesine altı alan sözcüğü eklenerek kurulur.
DUR_TR = durak_oku('durak_g2_tr.txt')
DUR_EN = durak_oku('durak_bulut_en.txt') | DOMAIN_EN


def kucult(m):
    """Türkçe için koşullu küçültme: İ → i, I → ı."""
    return m.replace('İ', 'i').replace('I', 'ı').replace('̇', '').lower()


def g2(a, b, c, d):
    """a: bantta sıklık · b: bant dışında · c: bandın hacmi · d: bant dışının hacmi"""
    e1 = c * (a + b) / (c + d)
    e2 = d * (a + b) / (c + d)
    g = 0.0
    if a > 0: g += a * math.log(a / e1)
    if b > 0: g += b * math.log(b / e2)
    g *= 2
    return g if (a / c) > (b / d) else -g


def veriyi_oku(yol):
    import openpyxl
    wb = openpyxl.load_workbook(yol, read_only=True, data_only=True)
    ws = wb[wb.sheetnames[0]]
    it = ws.iter_rows(values_only=True)
    bas = [str(c) for c in next(it)]
    IX = {a: i for i, a in enumerate(bas)}
    out = []
    for r in it:
        d = {a: r[IX[a]] for a in bas}
        t = (d.get('comment') or '').strip()
        if d.get('language_final') not in ('tr', 'en'): continue
        if not t or KINDLE.search(t): continue
        if d.get('rating_numeric_raw') in (None, ''): continue
        out.append({'dil': d['language_final'], 'metin': t,
                    'puan': int(float(d['rating_numeric_raw']))})
    return out


def sayaclar(kayitlar, dil, mor=None):
    """Her yorum için bir sayaç döndürür. Türkçede lemma, İngilizcede yüzey biçimi."""
    onbellek = {}
    cikti = []
    for k in kayitlar:
        c = collections.Counter()
        if dil == 'tr':
            for w in TOK_TR.findall(kucult(k['metin'])):
                if w not in onbellek:
                    try:
                        a = mor.analyze_and_disambiguate(w)
                        onbellek[w] = kucult(a[0].best_analysis.item.lemma) if a else w
                    except Exception:
                        onbellek[w] = w
                lem = onbellek[w].strip("'")
                if len(lem) >= 3 and lem not in DUR_TR:
                    c[lem] += 1
        else:
            for w in TOK_EN.findall(k['metin'].lower()):
                w = w.strip("'")
                if w and w not in DUR_EN:
                    c[w] += 1
        cikti.append(c)
    return cikti


def tablo(kayitlar, dil, ad, mor=None):
    per = {p: sayaclar([k for k in kayitlar if k['dil'] == dil and k['puan'] == p], dil, mor)
           for p in (1, 2, 3, 4, 5)}
    TUM = {p: sum(per[p], collections.Counter()) for p in per}
    H   = {p: sum(TUM[p].values()) for p in per}
    DOC = {p: collections.Counter(w for c in per[p] for w in c) for p in per}
    top = sum(H.values())

    print('\n' + '=' * 76)
    birim = 'lemma' if dil == 'tr' else 'sözcük'
    print(f'{ad} — puan düzeyine göre ayırt edici {birim}ler')
    print(f'ölçüt: G² · asgari sıklık {ASGARI_SIKLIK} · en az {ASGARI_YAYILIM} farklı yorum')

    for p in (1, 2, 3, 4, 5):
        c = H[p]; d = top - c
        B = collections.Counter()
        for q in TUM:
            if q != p: B.update(TUM[q])
        sk = []
        for w, a in TUM[p].items():
            if a < ASGARI_SIKLIK or DOC[p][w] < ASGARI_YAYILIM: continue
            g = g2(a, B[w], c, d)
            if g > 0: sk.append((g, w, a, B[w], DOC[p][w]))
        sk.sort(reverse=True)
        print(f'\n  {p} YILDIZ  (n={len(per[p])} yorum, {c} {birim})')
        if not sk:
            print('    eşiği geçen birim yok'); continue
        for g, w, a, b, dc in sk[:EN_UZUN]:
            print(f'    {w:18s} G²={g:5.1f}   bantta {a:4d} · diğerlerinde {b:4d} · {dc} yorumda')

## 4. Tabloların üretilmesi

In [ ]:
K = veriyi_oku(DOSYA)
print('puanlanmış yorum: %d (tr %d, en %d)'
      % (len(K), sum(1 for k in K if k['dil'] == 'tr'),
         sum(1 for k in K if k['dil'] == 'en')))

from zemberek import TurkishMorphology
mor = TurkishMorphology.create_with_defaults()

tablo(K, 'en', 'İNGİLİZCE KORPUS')
tablo(K, 'tr', 'TÜRKÇE KORPUS', mor)

## Bilinen sınırlar

Zemberek özel adları ortak adlardan ayırmaz. *galip* ve *rüya* sözlük
birimlerinde roman kişilerinin adları, aynı yazılışa sahip ortak adlarla
birleşir. Belirsizlik giderme de hatasız değildir. Bu nedenle makaleye giren
her sözcük ayrıca bağlamlı dizinle denetlenmiştir.

Çok sayıda sözcük aynı anda sınandığı için G² burada bir anlamlılık kararı
olarak değil, aşırı temsili sıraya dizen bir ölçü olarak kullanılmıştır.